# Árbol de decisión para predicción de aprobación de estudiantes con Pipeline + MLflow

Este notebook entrena un modelo de árbol de decisión usando el dataset `estudiantes.csv`.

## 1. Importación de librerías

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from mlflow.models.signature import infer_signature

import mlflow
import mlflow.sklearn


## 2. Configuración de MLflow



In [2]:
mlflow.set_tracking_uri("http://127.0.0.1:9090")
mlflow.set_experiment("prediccion_aprobacion")


2026/05/20 09:53:23 INFO mlflow.tracking.fluent: Experiment with name 'prediccion_aprobacion' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1779288803247, experiment_id='3', last_update_time=1779288803247, lifecycle_stage='active', name='prediccion_aprobacion', tags={}, trace_location=None, workspace='default'>

## 3. Carga del dataset

In [3]:
df = pd.read_csv("data/estudiantes.csv", delimiter=",")
df.head()


,carrera,modalidad,beca,edad,promedio,asistencias,aprobado
0,Industrial,Presencial,Si,29,5.8,64,Si
1,Industrial,Hibrida,Si,27,6.6,51,No
2,Arquitectura,Presencial,Si,29,8.2,84,Si
3,Economia,Presencial,Si,29,6.6,67,No
4,Economia,Presencial,Si,24,5.1,72,No


In [4]:
print("Filas y columnas:", df.shape)
df.info()


Filas y columnas: (10000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   carrera      10000 non-null  object 
 1   modalidad    10000 non-null  object 
 2   beca         10000 non-null  object 
 3   edad         10000 non-null  int64  
 4   promedio     10000 non-null  float64
 5   asistencias  10000 non-null  int64  
 6   aprobado     10000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 547.0+ KB


## 4. Preparación de datos

La variable objetivo `y` se transforma:

- `yes` → `1`
- `no` → `0`

In [5]:
df["aprobado"] = df["aprobado"].map({"Si": 1, "No": 0})

X = df.drop(columns=["aprobado"])
y = df["aprobado"]

print("Columnas originales usadas para entrenamiento:")
print(X.columns.tolist())

print("\nDistribución de la variable objetivo:")
print(y.value_counts())


Columnas originales usadas para entrenamiento:
['carrera', 'modalidad', 'beca', 'edad', 'promedio', 'asistencias']

Distribución de la variable objetivo:
aprobado
0    5844
1    4156
Name: count, dtype: int64


## 5. Identificación de columnas categóricas y numéricas

El pipeline hará automáticamente el `OneHotEncoder` para las columnas categóricas y dejará pasar las columnas numéricas.


In [6]:
columnas_categoricas = X.select_dtypes(include=["object"]).columns.tolist()
columnas_numericas = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Columnas categóricas:")
print(columnas_categoricas)

print("\nColumnas numéricas:")
print(columnas_numericas)


Columnas categóricas:
['carrera', 'modalidad', 'beca']

Columnas numéricas:
['edad', 'promedio', 'asistencias']


## 6. División de datos

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

Entrenamiento: (8000, 6)
Prueba: (2000, 6)


## 7. Creación del Pipeline

In [27]:
# Parámetros del árbol
criterion = "entropy"
max_depth = 2
min_samples_split = 60
min_samples_leaf = 24
random_state = 42

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas)
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("modelo", DecisionTreeClassifier(
            criterion=criterion,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=random_state
        ))
    ]
)

pipeline


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['carrera', 'modalidad',
                                                   'beca']),
                                                 ('num', 'passthrough',
                                                  ['edad', 'promedio',
                                                   'asistencias'])])),
                ('modelo',
                 DecisionTreeClassifier(criterion='entropy', max_depth=2,
                                        min_samples_leaf=24,
                                        min_samples_split=60,
                                        random_state=42))])

## 8. Entrenamiento, evaluación y registro en MLflow

El modelo se registrará con el nombre:
```text
prediccion_aprobacion
```

Luego se podrá cargar desde Streamlit con:
```python
mlflow.sklearn.load_model("models:/prediccionAprobacion/1")
```


In [28]:
with mlflow.start_run(run_name="prediccion_aprobacion") as run:

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Parámetros del algoritmo
    mlflow.log_param("modelo", "DecisionTreeClassifier")
    mlflow.log_param("criterion", criterion)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("min_samples_split", min_samples_split)
    mlflow.log_param("min_samples_leaf", min_samples_leaf)
    mlflow.log_param("random_state", random_state)

    # Parámetros del experimento
    mlflow.log_param("dataset", "estudiantes.csv")
    mlflow.log_param("target", "y")
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("stratify", True)
    mlflow.log_param("removed_column", "duration")
    mlflow.log_param("categorical_encoder", "OneHotEncoder")
    mlflow.log_param("handle_unknown", "ignore")

    # Métricas
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    
    # Firma del modelo: ayuda a documentar columnas esperadas
    input_example = X_train.head(5)
    signature = infer_signature(input_example, pipeline.predict(input_example))

    # Guardar y registrar el pipeline completo
    mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="prediccion_aprobacion",
        registered_model_name="prediccionAprobacion",
        input_example=input_example,
        signature=signature
    )

    print("RUN ID:", run.info.run_id)
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1:", f1)
    print("\nReporte de clasificación:")


C:\bin\clase03\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/20 10:25:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/20 10:25:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these

Registered model 'prediccionAprobacion' already exists. Creating a new version of this model...
2026/05/20 10:25:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: prediccionAprobacion, version 10
Created version '10' of model 'prediccionAprobacion'.


RUN ID: 71958046a8ab4b3081aeb4a29ff7bf82
Accuracy: 0.8555
Precision: 0.8280871670702179
Recall: 0.8231046931407943
F1: 0.8255884127942064

Reporte de clasificación:
🏃 View run prediccion_aprobacion at: http://127.0.0.1:9090/#/experiments/3/runs/71958046a8ab4b3081aeb4a29ff7bf82
🧪 View experiment at: http://127.0.0.1:9090/#/experiments/3
